In [1]:
import os, random, joblib, json
import numpy as np, pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.layers import Layer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from sklearn.metrics import roc_auc_score

In [25]:
# ---------------- CONFIG ----------------
CSV_PATH = "final_clean_dataset_cleaned_v3.csv"   # ensure this file is in the same folder
CATALOG_PATH = "habit_catalog_clean.json"
MODEL_DIR = "deepfm_model"
os.makedirs(MODEL_DIR, exist_ok=True)
PREPROC_PATH = os.path.join(MODEL_DIR, "preproc.joblib")
MODEL_PATH = os.path.join(MODEL_DIR, "model.keras")

SEED = 42
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

In [26]:
# ---------------- LOAD ----------------
print("Loading CSV:", CSV_PATH)
df = pd.read_csv(CSV_PATH)
print("Rows:", len(df))
if "label" not in df.columns:
    raise RuntimeError("label missing in CSV; run cleaning step first")

Loading CSV: final_clean_dataset_cleaned_v3.csv
Rows: 20000


In [27]:
# ---------------- SCHEMA ----------------
numeric_cols = ["risk_score","prediction","emotion_score","screen_time","unlocks","sleep_hours","steps_last_24h","popularity_prior"]
categorical_cols = ["habit_id","category","dopamine_level","difficulty","time_min","indoor","required_device","dominant_emotion"]

# ---------------- CLEAN ----------------
for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(df[c].median())

for c in categorical_cols:
    if c in df.columns:
        df[c] = df[c].astype(str).fillna("unknown")

In [28]:
# ---------------- ENCODE ----------------
encoders = {}
for c in categorical_cols:
    if c not in df.columns:
        continue
    le = LabelEncoder()
    df[c] = le.fit_transform(df[c])
    encoders[c] = le

# ---------------- SCALE ----------------
existing_numeric = [c for c in numeric_cols if c in df.columns]
scaler = None
if existing_numeric:
    scaler = StandardScaler()
    df[existing_numeric] = scaler.fit_transform(df[existing_numeric])

# save preproc
preproc = {"numeric_cols": existing_numeric, "categorical_cols": [c for c in categorical_cols if c in df.columns], "encoders": encoders, "scaler": scaler}
joblib.dump(preproc, PREPROC_PATH)
print("Saved preproc ->", PREPROC_PATH)

Saved preproc -> deepfm_model\preproc.joblib


In [29]:
# ---------------- PREPARE KERAS INPUTS ----------------
y = df["label"].astype(int).values
X_cat = {c: df[c].values for c in preproc["categorical_cols"]}
X_num = df[preproc["numeric_cols"]].values if preproc["numeric_cols"] else None

train_idx, val_idx = train_test_split(np.arange(len(df)), test_size=0.2, random_state=SEED, stratify=y)

def build_inputs(idx):
    out = {}
    for c in preproc["categorical_cols"]:
        out[c] = X_cat[c][idx]
    if preproc["numeric_cols"]:
        out["numeric_input"] = X_num[idx]
    return out

train_in = build_inputs(train_idx)
val_in = build_inputs(val_idx)

def to_keras_inputs(inp):
    out = {}
    for c in preproc["categorical_cols"]:
        out[f"inp_{c}"] = inp[c].astype("int32")
    if preproc["numeric_cols"]:
        out["numeric_input"] = inp["numeric_input"].astype("float32")
    return out

train_keras = to_keras_inputs(train_in)
val_keras = to_keras_inputs(val_in)
y_train = y[train_idx]
y_val = y[val_idx]

In [30]:
# ---------------- MODEL ----------------
class FMLayer(Layer):
    def call(self, inputs):
        summed = tf.reduce_sum(inputs, axis=1)
        summed_square = tf.square(summed)
        squared = tf.square(inputs)
        squared_sum = tf.reduce_sum(squared, axis=1)
        return 0.5 * tf.reduce_sum(summed_square - squared_sum, axis=1, keepdims=True)

EMB_DIM = 8
inputs = {}
emb_fm_list = []
emb_dnn_list = []

for c in preproc["categorical_cols"]:
    vocab = int(df[c].nunique()) + 1
    inp = Input(shape=(1,), name=f"inp_{c}")
    inputs[c] = inp
    emb = layers.Embedding(input_dim=vocab, output_dim=EMB_DIM, name=f"emb_{c}")(inp)
    emb_fm_list.append(emb)
    emb_dnn_list.append(layers.Reshape((EMB_DIM,))(emb))

if preproc["numeric_cols"]:
    num_inp = Input(shape=(len(preproc["numeric_cols"]),), name="numeric_input")
    inputs["numeric_input"] = num_inp
    num_fm = layers.Dense(EMB_DIM)(num_inp)
    num_fm = layers.Reshape((1, EMB_DIM))(num_fm)
    emb_fm_list.append(num_fm)
    emb_dnn_list.append(num_inp)

fm_tensor = layers.Concatenate(axis=1)(emb_fm_list)
fm_out = FMLayer()(fm_tensor)
linear_logit = layers.Dense(1)(layers.Concatenate()(emb_dnn_list))
dnn_input = layers.Concatenate()(emb_dnn_list)
x = layers.Dense(128, activation="relu")(dnn_input)
x = layers.Dropout(0.2)(x)
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.2)(x)
dnn_logit = layers.Dense(1)(x)
final_logit = layers.Add()([fm_out, linear_logit, dnn_logit])
output = layers.Activation("sigmoid")(final_logit)
model = Model(inputs=list(inputs.values()), outputs=output)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=[tf.keras.metrics.AUC(name="auc")])
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ inp_habit_id        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_category        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_dopamine_level  │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_difficulty      │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_time_min        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_indoor          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_required_device │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_dominant_emoti… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_habit_id        │ (None, 1, 8)      │      2,408 │ inp_habit_id[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_category        │ (None, 1, 8)      │         32 │ inp_category[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_dopamine_level  │ (None, 1, 8)      │         32 │ inp_dopamine_lev… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_difficulty      │ (None, 1, 8)      │         48 │ inp_difficulty[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_time_min        │ (None, 1, 8)      │         96 │ inp_time_min[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_indoor          │ (None, 1, 8)      │         24 │ inp_indoor[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_required_device │ (None, 1, 8)      │         32 │ inp_required_dev… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_dominant_emoti… │ (None, 1, 8)      │        232 │ inp_dominant_emo… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numeric_input       │ (None, 8)         │          0 │ -               

 Total params: 20,714 (80.91 KB)

 Trainable params: 20,714 (80.91 KB)

 Non-trainable params: 0 (0.00 B)

In [31]:
# ---------------- TRAIN ----------------
cw = class_weight.compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weights = {0: float(cw[0]), 1: float(cw[1])}
print("Class weights:", class_weights)

callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_auc", patience=4, mode="max", restore_best_weights=True)]
history = model.fit(train_keras, y_train, validation_data=(val_keras, y_val), epochs=25, batch_size=512, class_weight=class_weights, callbacks=callbacks, verbose=2)


Class weights: {0: 0.735023888276369, 1: 1.563721657544957}
Epoch 1/25
32/32 - 6s - 193ms/step - auc: 0.6856 - loss: 0.6320 - val_auc: 0.7859 - val_loss: 0.5571
Epoch 2/25
32/32 - 0s - 7ms/step - auc: 0.9030 - loss: 0.4371 - val_auc: 0.9471 - val_loss: 0.3083
Epoch 3/25
32/32 - 0s - 7ms/step - auc: 0.9639 - loss: 0.2417 - val_auc: 0.9748 - val_loss: 0.2144
Epoch 4/25
32/32 - 0s - 6ms/step - auc: 0.9832 - loss: 0.1685 - val_auc: 0.9859 - val_loss: 0.1568
Epoch 5/25
32/32 - 0s - 6ms/step - auc: 0.9897 - loss: 0.1281 - val_auc: 0.9907 - val_loss: 0.1233
Epoch 6/25
32/32 - 0s - 6ms/step - auc: 0.9924 - loss: 0.1053 - val_auc: 0.9924 - val_loss: 0.1089
Epoch 7/25
32/32 - 0s - 6ms/step - auc: 0.9941 - loss: 0.0914 - val_auc: 0.9935 - val_loss: 0.0985
Epoch 8/25
32/32 - 0s - 6ms/step - auc: 0.9949 - loss: 0.0840 - val_auc: 0.9942 - val_loss: 0.0940
Epoch 9/25
32/32 - 0s - 6ms/step - auc: 0.9958 - loss: 0.0755 - val_auc: 0.9948 - val_loss: 0.0871
Epoch 10/25
32/32 - 0s - 6ms/step - auc: 0.9963

In [32]:
# ---------------- SAVE ----------------
model.save(MODEL_PATH)
joblib.dump(preproc, PREPROC_PATH)
print("Saved model ->", MODEL_PATH)

Saved model -> deepfm_model\model.keras


In [33]:
# ---------------- INFERENCE ----------------
def safe_map_series(series, le):
    classes = le.classes_
    has_unknown = "unknown" in classes
    out = []
    for v in series:
        v_s = str(v)
        try:
            out.append(int(le.transform([v_s])[0]))
        except:
            if has_unknown:
                out.append(int(np.where(classes=="unknown")[0][0]))
            else:
                out.append(0)
    return np.array(out, dtype="int32")

def build_candidates(user_row, catalog_df):
    df_user = pd.DataFrame([user_row])
    catalog = catalog_df.copy()
    if "recommended_for_state" in catalog.columns:
        catalog = catalog.drop(columns=["recommended_for_state"])
    df_user_rep = pd.concat([df_user] * len(catalog), ignore_index=True)
    return pd.concat([df_user_rep.reset_index(drop=True), catalog.reset_index(drop=True)], axis=1)

def preprocess_inf(df_rows, pre):
    encs = pre["encoders"]
    scaler = pre["scaler"]
    cat_cols = pre["categorical_cols"]
    num_cols = pre["numeric_cols"]
    dfp = df_rows.copy()
    for c in cat_cols:
        if c not in dfp.columns: dfp[c] = "unknown"
    for n in num_cols:
        if n not in dfp.columns: dfp[n] = 0.0
    for c in cat_cols:
        le = encs[c]
        dfp[c] = dfp[c].astype(object).fillna("unknown")
        dfp[c] = safe_map_series(dfp[c].tolist(), le)
    if num_cols and scaler is not None:
        dfp[num_cols] = scaler.transform(dfp[num_cols])
    inp = {}
    for c in cat_cols:
        inp[f"inp_{c}"] = dfp[c].astype("int32").values
    if num_cols:
        inp["numeric_input"] = dfp[num_cols].astype("float32").values
    return inp, dfp

In [35]:
# quick recommend example
catalog_df = pd.read_json(CATALOG_PATH)

user_snapshot_1 = {"risk_score":0.970,"prediction":1,"dominant_emotion":"Sadness","emotion_score":0.967,"screen_time":500,"unlocks":55,"sleep_hours":9,"steps_last_24h":1800}
user_snapshot_2 = {"risk_score":0.22,"prediction":0,"dominant_emotion":"joy","emotion_score":-0.61,"screen_time":195,"unlocks":32,"sleep_hours":7.6,"steps_last_24h":6200}

user_snapshot_demo = user_snapshot_2 = {"risk_score":0.30,"prediction":0,"dominant_emotion":"Embarrassment","emotion_score":0.970,"screen_time":320,"unlocks":38,"sleep_hours":7,"steps_last_24h":2600}

pre = joblib.load(PREPROC_PATH)
mdl = tf.keras.models.load_model(MODEL_PATH, custom_objects={"FMLayer": FMLayer})

def recommend(user_row, catalog_df, top_k=5):
    df_cand = build_candidates(user_row, catalog_df)
    inp, df_proc = preprocess_inf(df_cand, pre)
    scores = mdl.predict(inp, batch_size=512).ravel()
    df_proc["score"] = scores
    if "heuristic_score" in df_proc.columns:
        df_proc["score"] = df_proc["score"] + 0.12 * df_proc["heuristic_score"]
    df_sorted = df_proc.sort_values("score", ascending=False)
    habit_le = pre["encoders"]["habit_id"]
    df_sorted["habit_id_original"] = habit_le.inverse_transform(df_sorted["habit_id"].astype(int))
    return df_sorted.head(top_k)[["habit_id_original","score","category","difficulty","time_min"]]

# print("Top 5 for snapshot 1:")
# print(recommend(user_snapshot_1, catalog_df, top_k=5).to_string(index=False))
# print("Top 5 for snapshot 2:")
# print(recommend(user_snapshot_2, catalog_df, top_k=5).to_string(index=False))

print("Top 5 for snapshot 2:")
print(recommend(user_snapshot_demo, catalog_df, top_k=5).to_string(index=False))

Top 5 for snapshot 2:


C:\Users\r8456\AppData\Local\Temp\ipykernel_23892\3172114892.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dfp[c] = dfp[c].astype(object).fillna("unknown")
C:\Users\r8456\AppData\Local\Temp\ipykernel_23892\3172114892.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dfp[c] = dfp[c].astype(object).fillna("unknown")
C:\Users\r8456\AppData\Local\Temp\ipykernel_23892\3172114892.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) ins

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 401ms/step
habit_id_original    score  category  difficulty  time_min
              115 0.999952         1           0         0
              295 0.999947         1           0         0
              203 0.999939         1           1         9
               55 0.999939         1           0         6
              190 0.999912         1           1         9
